In [1]:
import json
import re
import string
from collections import Counter
from pathlib import Path

import numpy as np
from datasets import load_dataset

from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import OllamaLLM
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate


c:\Users\HIlla\Documents\vanilla rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GENERATOR_MODEL = "llama3.1:8b"
TOP_K = 5
N_EXAMPLES = 2000   # Para probar. Luego sube a 200, 500, 1000 o None

DATASET_NAME = "coqa"
DATASET_SPLIT = "dev"
MULTIPLE_ANSWERS = True

CHROMA_DIR = Path("./chroma_db")
CHROMA_DIR.mkdir(exist_ok=True)

RESULTS_FILE = Path("./resultados_coqa.json")

COQA_SYSTEM_PROMPT = (
    "You are a helpful assistant who will try to answer the following question "
    "to the best of your abilities. Use only on the given context and conversation "
    "history and do not use any assumptions or external information. Make the answers "
    "as direct as possible without using any redundant information and without using "
    "full sentences. Indicate if you cannot find the answer based on the context."
)

In [ ]:
def normalizar(s: str) -> str:
    s = s.lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(ch for ch in s if ch not in string.punctuation)
    return " ".join(s.split())


def f1_score(pred: str, gold: str) -> float:
    pred_toks = normalizar(pred).split()
    gold_toks = normalizar(gold).split() #palabras de la respuesta correcta con las que comparas la respuesta

    if not pred_toks or not gold_toks:
        return float(pred_toks == gold_toks) 

    comunes = sum((Counter(pred_toks) & Counter(gold_toks)).values())
    if comunes == 0:
        return 0.0

    prec = comunes / len(pred_toks)
    rec = comunes / len(gold_toks)
    return 2 * prec * rec / (prec + rec)


In [ ]:
def construir_historial_conversacional(messages):

    # solo user y assistant
    mensajes_validos = [m for m in messages if m["role"] != "system"]

    if not mensajes_validos:
        return ""

    # detectar si el ultimo mensaje es del usuario entoncs es la pregunta actual
    if mensajes_validos[-1]["role"] == "user":
        # e elimina la última pregunta para evitar duplicarla en el prompt
        mensajes_historial = mensajes_validos[:-1]
    else:
        # Si no es una pregunta, se usa todo como historial
        mensajes_historial = mensajes_validos

    partes = []
    for m in mensajes_historial:
        role = m["role"].capitalize() 
        partes.append(f"{role}: {m['content']}")

    return "\n".join(partes)

def construir_query_conversacional(messages): #query para el retrival

    partes = []
    for m in messages:
        if m["role"] == "system":
            continue
        partes.append(m["content"])
    return " ".join(partes)

In [ ]:
print(f"▶ Cargando {DATASET_NAME.upper()} ({DATASET_SPLIT})...")

ds = load_dataset("nvidia/ChatRAG-Bench", DATASET_NAME, split=DATASET_SPLIT)

if N_EXAMPLES:
    ds = ds.select(range(min(N_EXAMPLES, len(ds))))

# Pregunta actual de cada ejemplo
preguntas = []

# Historial conversacional (para el LLM)
historiales = []

# Conversación completa serializada (para el retriever)
queries_conversacionales = []

# Respuestas correctas
respuestas_gold = []

# ID 
gold_ctx_ids = []

documentos = []                 # lista final de Document para Chroma
ctx_vistos = {}                 # evita duplicar documentos repetidos

# Recorre cada ejemplo del dataset
for row in ds:

    mensajes = row["messages"]
    mensajes_usuario = [m for m in mensajes if m["role"] == "user"]
    ultima = mensajes_usuario[-1]

    pregunta_actual = ultima["content"]


    # Construye el historial conversacional (para el LLM)
    historial = construir_historial_conversacional(mensajes)

    # Construye la query conversacional (para retrieval)
    query_conv = construir_query_conversacional(mensajes)


    # Guarda la pregunta actual
    preguntas.append(pregunta_actual)
    historiales.append(historial)
    queries_conversacionales.append(query_conv)
    respuestas_gold.append(row["answers"])

    # Toma el primer contexto como documento correcto
    gold_texto = row["ctxs"][0]["text"]

    # Si este documento aún no fue guardado
    if gold_texto not in ctx_vistos:

        # Crear un ID único (doc_0, doc_1, ...)
        doc_id = f"doc_{len(documentos)}"

        # Guardarlo en el diccionario para evitar duplicados
        ctx_vistos[gold_texto] = doc_id

        # Crear objeto Document (para Chroma)
        documentos.append(
            Document(
                page_content=gold_texto,       # texto del documento
                metadata={"doc_id": doc_id}    # ID del documento
            )
        )

    # Guardar el ID del documento correcto para este ejemplo
    gold_ctx_ids.append(ctx_vistos[gold_texto])


    for c in row["ctxs"][1:]:

        if c["text"] not in ctx_vistos:

            doc_id = f"doc_{len(documentos)}"

            # Guardar en el diccionario
            ctx_vistos[c["text"]] = doc_id

            documentos.append(
                Document(
                    page_content=c["text"],
                    metadata={"doc_id": doc_id}
                )
            )

print(f"  {len(preguntas)} preguntas | {len(documentos)} contextos únicos")

▶ Cargando COQA (dev)...


  2000 preguntas | 129 contextos únicos


In [ ]:
# VECTOR STORE

print("▶ Creando embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)

collection_name = "coqa_collection"

if any(CHROMA_DIR.iterdir()):
    print("▶ Cargando Chroma existente...")
    vectorstore = Chroma(
        persist_directory=str(CHROMA_DIR),
        embedding_function=embeddings,
        collection_name=collection_name,
    )
else:
    print("▶ Indexando documentos en Chroma...")
    vectorstore = Chroma.from_documents(
        documents=documentos,
        embedding=embeddings,
        collection_name=collection_name,
        collection_metadata={"hnsw:space": "cosine"},
        persist_directory=str(CHROMA_DIR),
    )

retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K}) #convierte tu base de datos en un sistema capaz de buscar los documentos más relevantes para una consulta

▶ Creando embeddings...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7283.99it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


▶ Indexando documentos en Chroma...


In [7]:

# GENERADOR
print(f"▶ Conectando con Ollama: {GENERATOR_MODEL}")
llm = OllamaLLM(
    model=GENERATOR_MODEL,
    temperature=0,
    num_predict=1000,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", COQA_SYSTEM_PROMPT),
    ("user",
     "Context:\n{context}\n\n"
     "Conversation history:\n{history}\n\n"
     "Question: {question}"),
])

chain = prompt | llm


def generar_respuesta(pregunta: str, history: str, docs: list[Document]) -> str:
    contexto = "\n\n".join(
        f"[{i+1}] {d.page_content}" for i, d in enumerate(docs)
    )
    return chain.invoke({
        "context": contexto,
        "history": history if history.strip() else "None",
        "question": pregunta
    }).strip()


▶ Conectando con Ollama: llama3.1:8b


In [ ]:

print("\n▶ Verificación inicial del retriever…")

# Usa la primera query conversacional para recuperar documentos
test_docs = retriever.invoke(queries_conversacionales[0])

# Extrae los ids de los documentos recuperados
test_ids = [d.metadata["doc_id"] for d in test_docs]

# Muestra la pregunta, el gold esperado y los ids recuperados
print(f"  Pregunta: {preguntas[0]}")
print(f"  Gold esperado: {gold_ctx_ids[0]}")
print(f"  Top-{TOP_K}: {test_ids}")

# Verifica si el documento correcto está en la lista recuperada
if gold_ctx_ids[0] in test_ids:
    rank = test_ids.index(gold_ctx_ids[0]) + 1
    print(f"  ✓ Gold en posición {rank}")
else:
    print(f"  ⚠ Gold no está en Top-{TOP_K}")


▶ Verificación inicial del retriever…
  Pregunta: What color was Cotton?
  Gold esperado: doc_0
  Top-5: ['doc_0', 'doc_127', 'doc_47', 'doc_69', 'doc_39']
  ✓ Gold en posición 1


In [9]:
print("\n▶ Evaluando…")
f1s = []
mrrs = []
detalles = []

for i, pregunta in enumerate(preguntas):
    history = historiales[i]
    query_conv = queries_conversacionales[i]

    docs = retriever.invoke(query_conv)
    top_ids = [d.metadata["doc_id"] for d in docs]

    respuesta = generar_respuesta(pregunta, history, docs)
    gold_id = gold_ctx_ids[i]

    if MULTIPLE_ANSWERS:
        f1 = max(f1_score(respuesta, g) for g in respuestas_gold[i])
    else:
        f1 = f1_score(respuesta, respuestas_gold[i][0])

    mrr = 0.0
    for rank, did in enumerate(top_ids, start=1):
        if did == gold_id:
            mrr = 1.0 / rank
            break

    f1s.append(f1)
    mrrs.append(mrr)

    detalles.append({
        "idx": i,
        "question": pregunta,
        "history": history,
        "generated_answer": respuesta,
        "gold_answers": respuestas_gold[i],
        "gold_id": gold_id,
        "top_ids": top_ids,
        "f1": f1,
        "mrr5": mrr,
    })

    if (i + 1) % 10 == 0:
        print(
            f"  {i+1}/{len(preguntas)}  "
            f"F1={np.mean(f1s) * 100:.2f}  "
            f"MRR@5={np.mean(mrrs) * 100:.2f}"
        )


▶ Evaluando…
  10/2000  F1=61.87  MRR@5=100.00
  20/2000  F1=74.15  MRR@5=100.00
  30/2000  F1=66.61  MRR@5=96.67
  40/2000  F1=68.77  MRR@5=93.83
  50/2000  F1=71.00  MRR@5=95.07
  60/2000  F1=71.17  MRR@5=95.89
  70/2000  F1=66.67  MRR@5=96.48
  80/2000  F1=69.34  MRR@5=94.83
  90/2000  F1=71.40  MRR@5=95.41
  100/2000  F1=73.04  MRR@5=95.87
  110/2000  F1=74.51  MRR@5=96.24
  120/2000  F1=74.58  MRR@5=96.56
  130/2000  F1=73.63  MRR@5=96.82
  140/2000  F1=74.53  MRR@5=97.05
  150/2000  F1=75.05  MRR@5=97.24
  160/2000  F1=75.33  MRR@5=96.79
  170/2000  F1=74.42  MRR@5=95.80
  180/2000  F1=75.43  MRR@5=96.04
  190/2000  F1=74.96  MRR@5=95.98
  200/2000  F1=75.17  MRR@5=96.18
  210/2000  F1=74.55  MRR@5=96.37
  220/2000  F1=74.19  MRR@5=94.94
  230/2000  F1=73.61  MRR@5=95.16
  240/2000  F1=74.20  MRR@5=95.08
  250/2000  F1=74.34  MRR@5=95.28
  260/2000  F1=73.64  MRR@5=95.08
  270/2000  F1=73.45  MRR@5=95.26
  280/2000  F1=73.27  MRR@5=95.07
  290/2000  F1=73.72  MRR@5=95.24
  300/2

In [10]:

f1_final = float(np.mean(f1s) * 100)
mrr_final = float(np.mean(mrrs) * 100)

print("\n" + "═" * 58)
print(f"Vanilla RAG sobre CoQA ({len(preguntas)} ejemplos)")
print("═" * 58)
print(f"F1        = {f1_final:.2f}")
print(f"MRR@5     = {mrr_final:.2f}")

print("\nComparación con paper:")
print(f"F1        → : {f1_final:.2f} | paper: 58.3")
print(f"MRR@5     → : {mrr_final:.2f} | paper: 72.5")

resultado_json = {
    "dataset": DATASET_NAME,
    "split": DATASET_SPLIT,
    "n_examples": len(preguntas),
    "embedding_model": EMBEDDING_MODEL,
    "generator_model": GENERATOR_MODEL,
    "top_k": TOP_K,
    "f1": f1_final,
    "mrr5": mrr_final,
    "paper_reference": {
        "f1": 58.3,
        "mrr5": 72.5
    },
    "details": detalles,
}

with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    json.dump(resultado_json, f, ensure_ascii=False, indent=2)


══════════════════════════════════════════════════════════
Vanilla RAG sobre CoQA (2000 ejemplos)
══════════════════════════════════════════════════════════
F1        = 73.08
MRR@5     = 94.56

Comparación con paper:
F1        → : 73.08 | paper: 58.3
MRR@5     → : 94.56 | paper: 72.5
